In [ ]:
import kagglehub
import pandas as pd


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")


print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(f"{path}/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

# RangeIndex: 20001 entries, 0 to 20000
# Columns: 189 entries, P_2 to Target

# alot of cols have more than 70% data missing so drop cols
# df = df.drop("D_142", axis=1)

# col B_2 has only 88 missing so drop rows
df = df.dropna(subset=["B_2"])

# cols D_143, D_144, D_145 have about 2k missing so impute with mean
cols = ["D_143", "D_144", "D_145", "P_2", "D_141"]
for col in cols:
  df[col] = df[col].fillna(df[col].mean())

missing_data = df.isnull().sum()

# dropped cols which had more than 65-70% entries missing
cols = missing_data[missing_data > 10000]
drop_cols = cols.index
drop_cols


df.drop(drop_cols, axis=1, inplace=True)
df.info()

# checked and most were around 3-5k entries msising so better to impute them with mean
missing_data = df.isnull().sum()

remaining_cols = missing_data[missing_data < 10000]
rem_cols = remaining_cols.index
for col in rem_cols:
  df[col] = df[col].fillna(df[col].mean())

# final check no missing values remainnig
missing_data = df.isnull().sum()
print(missing_data.to_string())




In [ ]:
# Task 2: Write your code here:
df = df.drop_duplicates()

In [ ]:
# Task 3: Write your code here:

# Encode categorical variables if needed

# no categorical cols
categorical_cols = df.select_dtypes(include=["object"]).columns
categorical_cols


In [ ]:
# Task 4: Write your code here:

# Apply feature scaling to numerical features (Use StandardScaler)

from sklearn.preprocessing import StandardScaler


scaler = StandardScaler()
numerical_cols = df.select_dtypes(include=["float64", "int64"]).columns

cols_to_scale = df.columns.drop("Target")
df[cols_to_scale] = scaler.fit_transform(df[cols_to_scale])
df.head()



In [ ]:
# Task 5: Write your code here:

# Check for target imbalance and state if it is imbalanced or not

counts = df["Target"].value_counts()
counts

# Target
# 0	14678
# 1	5235

# clear imbalance and hence better to use stratify to have same ratio classes for the target


In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1)
y = df["Target"]

In [ ]:
# Task 2,3,4,5: Write your code here:
import torch.nn.functional as F
import torch

# better to use stratified k fold since there is imbalance in target
%pip install catboost
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = CatBoostClassifier(n_estimators=200, verbose=False)

model.fit(X, y)

# classification report is used to print out a comprehensive metric

for train_idx, val_idx in skf.split(X, y):


  X_fold_train, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
  y_fold_train, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

  model.fit(X_fold_train, y_fold_train)
  y_fold_pred = model.predict(X_fold_val)



  print(classification_report(y_fold_pred, y_fold_val))











In [ ]:
# Task 1: Write your code here:

import matplotlib.pyplot as plt

feature_cols = X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(100, 60))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

P_2

In [ ]:
# Task Bonus: Write your code here: